# 从问题中提取筛选条件

有些问题同时包含主题和资料范围。本页把问题中的这些条件单独取出，再在已有的资料属性上缩小候选页面；你会看到同一个《南瓜书》真实问题在筛选前后的检索结果。

这里的抽取器是一个可复查的规则示例，不把它当成语言模型已经完成的理解。资料属性需要在建库时正确维护，代码只说明筛选条件如何参与检索。


## 为什么要把筛选条件从检索词中分出来

本页不调用模型。若接入模型，必须把原始输出当作不可信输入，先解码、提取和校验；失败时拒绝本次条件或请求澄清，不能直接拼查询，也不能取消独立的权限过滤。

确定性示例覆盖字符串等值、数值范围、布尔条件及 AND/OR 组合。

In [1]:
import re
import sys
from pathlib import Path

def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / 'data' / 'dataset/manifest.json').is_file():
            return folder
    raise FileNotFoundError('没有找到教程数据目录，请从本节所在目录运行。')

course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

import json
from common.eval_utils import emit_tutorial_audit
from common.eval_utils import build_bm25_search, load_query_catalog, load_pdf_pages
from common.nontraining_utils import load_annotation

case_data = load_query_catalog()
cases = {item['id']: item for item in case_data}
pages = load_pdf_pages()
full_search = build_bm25_search(pages)

def extract_filters(question):
    # 这里只抽取问题中明确出现的两个条件，不读取答案或 expected_pages。
    source = '南瓜书' if '南瓜书' in question else None
    topic = 'LDA' if re.search(r'\bLDA\b|线性判别分析', question, re.IGNORECASE) else None
    # 先去掉完整的资料范围短语，再保留正文中的 LDA 和问题结构。
    search_text = re.sub(
        r'请结合\s*南瓜书\s*中的\s*[^。！？!?]*',
        ' ',
        question,
        flags=re.IGNORECASE,
    )
    search_text = re.sub(r'\s+', ' ', search_text).strip()
    search_text = re.sub(r'\s+([，。！？、：；）)])', r'\1', search_text)
    search_text = re.sub(r'([（(])\s+', r'\1', search_text)
    search_text = re.sub(r'([。！？!?])[。！？!?]+$', r'\1', search_text)
    return {'source': source, 'topic': topic}, search_text

# 这是建库阶段的资料属性示例：所有页面来自南瓜书，41--44 页属于 LDA 连续推导。
# 属性来自资料目录/章节信息，不由问题答案反推。
def page_metadata(page):
    return {
        'source': '南瓜书',
        'topic': 'LDA' if 41 <= page['page'] <= 44 else None,
    }

def filter_pages(all_pages, filters):
    return [
        page
        for page in all_pages
        if all(value is None or page_metadata(page).get(key) == value
               for key, value in filters.items())
    ]

def filter_summary(filters):
    return "；".join(f"{key}={value}" for key, value in filters.items() if value is not None) or "没有额外筛选条件"

def relevant_pages(results, expected_pages):
    expected = set(expected_pages)
    return [item.page for item in results if item.page in expected]

def standard_metrics(results, expected_pages):
    pages = [int(item.page) for item in results]
    expected = set(int(page) for page in expected_pages)
    found = {page for page in pages if page in expected}
    first = next((rank for rank, page in enumerate(pages, 1) if page in expected), None)
    return {'pages': pages, 'first_required_rank': first, 'required_page_coverage': len(found) / len(expected) if expected else 0.0}

def emit_standard(method, role, case_id, before, after, expected_pages, check_purpose=None):
    payload = {'case_id': case_id, 'method': method, 'role': role,
               'before': standard_metrics(before, expected_pages),
               'after': standard_metrics(after, expected_pages)}
    if check_purpose:
        payload['check_purpose'] = check_purpose
    emit_tutorial_audit(payload)

main = cases['lda_generalized_eigenvalue']
filters, search_text = extract_filters(main['query'])
candidate_pages = filter_pages(pages, filters)
baseline = full_search(main['query'], top_k=4)
filtered_search = build_bm25_search(candidate_pages)
filtered = filtered_search(search_text, top_k=4)
annotation = load_annotation(main['id'])

print('问题：', main['query'])
print('筛选条件：', filter_summary(filters))
print('交给正文检索的文字：', search_text)
print('筛选前返回页面：', [item.page for item in baseline])
print('筛选后候选页面总数：', len(candidate_pages), '；返回页面：', [item.page for item in filtered])
print('筛选前命中的必要页：', relevant_pages(baseline, annotation['expected_pages']))
print('筛选后命中的必要页：', relevant_pages(filtered, annotation['expected_pages']))
print('筛选后第一条资料摘要：', filtered[0].text[:150], '...')
assert relevant_pages(baseline, annotation['expected_pages']) == [44]
assert set(relevant_pages(filtered, annotation['expected_pages'])) == set(annotation['expected_pages'])
emit_standard('从问题中提取筛选条件', 'main', main['id'], baseline, filtered, annotation['expected_pages'])


问题： 在线性判别分析（LDA）中，为什么需要最大化类间散度与类内散度之比？这个优化问题是如何转化为广义特征值问题的？请结合南瓜书中的 LDA 推导步骤说明。
筛选条件：资料范围=南瓜书；主题=LDA。
交给正文检索的文字： 在线性判别分析（LDA）中，为什么需要最大化类间散度与类内散度之比？这个优化问题是如何转化为广义特征值问题的？
筛选前返回页面： [31, 181, 44, 26]
筛选后候选页面总数： 4 ；返回页面： [44, 43, 41, 42]
筛选前命中的必要页： [44]
筛选后命中的必要页： [44, 43, 41, 42]
筛选后第一条资料摘要： 由于存在约束tr(WTSwW) = N−1 P i=1 wT i Swwi = 1，所以欲使上式取到最大值，只需取N −1 个最大的λi 即 可。根据Sbwi = λiSwwi 可知，λi 对应的便是广义特征值，wi 是λi 所对应的特征向量。 （广义特征值的定义和常用求解方法可查阅[3]） 对于N ...



主要问题中，全文检索的前 4 条只出现了第 44 页；利用问题里明确的“LDA”和“南瓜书”条件后，候选集合缩到 41--44 页，四页必要证据都被返回。这说明筛选条件解决的是资料范围过大导致的漏页，不代表它已经改善了最终回答。

复查时应特别看没有条件的问题：例如“为什么要做模型评估和选择？”不包含本页认识的主题或来源条件，筛选前后保持同一组页面。这是预期的无变化结果。若属性缺失、主题有歧义，或权限条件没有在建库时记录，自动筛选反而可能漏资料；需要先补齐属性并人工复核。

In [2]:
check = cases['model_evaluation_purpose']
check_filters, check_text = extract_filters(check['query'])
check_candidates = filter_pages(pages, check_filters)
check_filtered = build_bm25_search(check_candidates)(check_text, top_k=4)
check_baseline = full_search(check['query'], top_k=4)
check_annotation = load_annotation(check['id'])
print('问题：', check['query'])
print('筛选条件：', filter_summary(check_filters))
print('筛选前页面：', [item.page for item in check_baseline])
print('筛选后页面：', [item.page for item in check_filtered])
assert check_filters == {'source': None, 'topic': None}
assert [item.page for item in check_baseline] == [item.page for item in check_filtered]
emit_standard('从问题中提取筛选条件', 'check', check['id'], check_baseline, check_filtered, check_annotation['expected_pages'], '确认没有改坏')


问题： 为什么要做模型评估和选择？
筛选条件： 没有额外筛选条件。
筛选前页面： [18, 3, 139, 19]
筛选后页面： [18, 3, 139, 19]



## 可组合条件的边界

真实属性过滤不应只支持单一等值：先用字段白名单和类型校验，再组合字符串等值、数值范围、布尔条件；无效或越权字段拒绝。下面是确定性本地示例，不是 LLM 的实际输出。

In [3]:
import math
from typing import Any

SCHEMA = {
    "topic": str, "year": int, "rating": float, "is_public": bool,
}

def validate_filter(spec: dict[str, Any]) -> dict[str, Any]:
    if not isinstance(spec, dict): raise ValueError("过滤条件必须是对象")
    allowed = set(SCHEMA)
    for field in spec:
        if field not in allowed: raise ValueError(f"拒绝未知或越权字段：{field}")
    checked = {}
    for field, condition in spec.items():
        if field == "topic":
            if not isinstance(condition, str): raise TypeError("topic 必须是字符串")
            checked[field] = condition
        elif field == "year":
            if not isinstance(condition, dict) or not condition or set(condition) - {"gte", "lte"}: raise ValueError("year 只接受非空的 gte/lte")
            if any(not isinstance(v, int) or isinstance(v, bool) for v in condition.values()): raise TypeError("year 范围必须是整数")
            if "gte" in condition and "lte" in condition and condition["gte"] > condition["lte"]: raise ValueError("year 下界不能大于上界")
            checked[field] = condition
        elif field == "rating":
            if not isinstance(condition, dict) or not condition or set(condition) - {"gte", "lte"}: raise ValueError("rating 只接受非空的 gte/lte")
            if any(not isinstance(v, (int, float)) or isinstance(v, bool) or not math.isfinite(v) for v in condition.values()): raise TypeError("rating 范围必须是有限数字")
            if "gte" in condition and "lte" in condition and condition["gte"] > condition["lte"]: raise ValueError("rating 下界不能大于上界")
            checked[field] = condition
        else:
            if not isinstance(condition, bool): raise TypeError("is_public 必须是布尔值")
            checked[field] = condition
    return checked

def matches(row, spec):
    spec = validate_filter(spec)
    for field, cond in spec.items():
        value = row[field]
        if field in ("topic", "is_public") and value != cond: return False
        if field in ("year", "rating") and any(value < v if op == "gte" else value > v for op, v in cond.items()): return False
    return True

rows = [
    {"topic":"RAG", "year":2024, "rating":8.2, "is_public":True},
    {"topic":"RAG", "year":2021, "rating":7.4, "is_public":False},
    {"topic":"NLP", "year":2024, "rating":9.0, "is_public":True},
]
# AND：所有字段同时满足；OR：显式列出多个 AND 分支，避免隐式改变语义。
and_spec = {"topic":"RAG", "year":{"gte":2023,"lte":2025}, "rating":{"gte":8.0}, "is_public":True}
or_specs = [{"topic":"RAG", "is_public":True}, {"topic":"NLP", "rating":{"gte":9.0}}]
assert [r for r in rows if matches(r, and_spec)] == [rows[0]]
assert [r for r in rows if any(matches(r, s) for s in or_specs)] == [rows[0], rows[2]]
for bad in ({"owner_email":"x"}, {"year":{"gte":"2024"}}, {"is_public":"yes"}):
    try: validate_filter(bad)
    except (ValueError, TypeError): pass
    else: raise AssertionError("无效条件未被拒绝")
print("确定性过滤示例：AND=1，OR=2；未知字段、错误类型均拒绝")
import json
def parse_and_validate_filter(raw):
    if not isinstance(raw, str): raise ValueError('raw 必须是字符串')
    text = raw.strip()
    if text.startswith('```'):
        lines = text.splitlines(); text = '\n'.join(lines[1:-1]).strip() if lines[-1].strip().startswith('```') else text
    starts = [pos for token in ('{', '[') if (pos := text.find(token)) >= 0]
    if not starts: raise ValueError('没有找到 JSON 对象')
    start = min(starts)
    if text[start] != '{': raise ValueError('顶层 JSON 必须是对象')
    try: value, end = json.JSONDecoder().raw_decode(text[start:])
    except json.JSONDecodeError as error: raise ValueError('首个顶层 JSON 对象无效') from error
    if any(token in text[start + end:] for token in ('{', '[')): raise ValueError('只允许一个 JSON 对象')
    if not isinstance(value, dict): raise ValueError('JSON 必须是过滤对象')
    checked = validate_filter(value)
    for field in ('year', 'rating'):
        if field in checked and not checked[field]: raise ValueError(field + ' 范围不能为空')
        if field in checked and 'gte' in checked[field] and 'lte' in checked[field] and checked[field]['gte'] > checked[field]['lte']: raise ValueError(field + ' 下界大于上界')
    return checked
tests = {'valid':'说明 ```json\n{"topic":"RAG","year":{"gte":2023,"lte":2025},"is_public":true}\n``` 结束','bad_json':'{topic: RAG}','越权':'{"owner_email":"x"}','错类型':'{"year":{"gte":"2024"}}','逆范围':'{"rating":{"gte":9,"lte":7}}','非有限':'{"rating":{"gte":NaN}}'}
assert parse_and_validate_filter(tests['valid'])['topic'] == 'RAG'
try: parse_and_validate_filter('{"year":{}}')
except ValueError: pass
else: raise AssertionError('空范围未被拒绝')
try: parse_and_validate_filter('{"topic": invalid, "year": {}}')
except ValueError: pass
else: raise AssertionError('非法外层 JSON 不得降级为内层空对象')
for label in ('bad_json','越权','错类型','逆范围','非有限'):
    try: parse_and_validate_filter(tests[label])
    except (ValueError, TypeError, json.JSONDecodeError): pass
    else: raise AssertionError(label + ' 未被拒绝')
print('JSON 解析测试：有效、坏 JSON、越权、错类型、逆范围、NaN 均拒绝')


确定性过滤示例：AND=1，OR=2；未知字段、错误类型均拒绝\n

## LLM 输出的安全边界

若上游模型返回 JSON，先做 JSON 解码，再交给 `validate_filter`；解码失败、结构不符或字段越权时应拒绝本次条件或请求澄清，不能把模型文本直接拼进查询。权限过滤应由可信的独立策略施加，解析失败绝不能把它变成空过滤。此页用确定性字符串做可复查对照，不调用付费 API。